# 1. 导入库与环境配置

本节目标：

- 搭建项目运行环境
- 导入所有将使用的核心库
- 检测 GPU / CPU 运行设备
- 为后续实验做可复现性配置

本项目以“数字图像处理”为核心，因此本节主要导入：

- **图像处理库**：OpenCV、scikit-image
- **数据处理库**：numpy、pandas
- **可视化库**：matplotlib、seaborn
- **深度学习库（后面 CNN 会用）**：PyTorch 或 TensorFlow（二选一，这里使用 PyTorch）
- **系统库**：os、glob、pathlib

重点：图像在计算机中以矩阵形式表达，因此 numpy + cv2 是本项目的最基本依赖。

下面开始正式导库与环境检测。


In [ ]:
# ================================================
# Section 1 — 导入库与环境配置
# ================================================

# ----- 基础库 -----
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['SimHei']  # 黑体：让中文标题正常显示
matplotlib.rcParams['axes.unicode_minus'] = False    # 解决负号显示为方块的问题

import os
import sys
import numpy as np
import pandas as pd

# ----- 图像处理库 -----
import cv2
from skimage import io, color, filters, feature

# ----- 可视化库 -----
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid", font_scale=1.2)  # 设置 seaborn 白网格风格和默认字号

# ----- 深度学习库（用于 CNN） -----
import torch
import torch.nn as nn
import torch.nn.functional as F

# ----- 实验可复现性 -----
import random

# 固定随机种子（保证每次结果一致）
seed = 42
np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():          # 只有有 CUDA 才去设 CUDA 的种子
    torch.cuda.manual_seed_all(seed)

# ================================================
# 检查 PyTorch 和 GPU 环境
# ================================================
print("PyTorch 版本:", torch.__version__)

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("CUDA 可用 ✓")
    print("GPU 名称:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("CUDA 不可用，使用 CPU（当前是 2.9.1+cpu 版本，这是正常的）")

print("当前计算设备:", device)

# ================================================
# 检查 OpenCV
# ================================================
print("OpenCV 版本:", cv2.__version__)

# 测试 matplotlib 图像输出是否正常工作
test_img = np.zeros((200, 200), dtype=np.uint8)
cv2.circle(test_img, (100, 100), 50, 255, 3)  # 在黑色画布上画白色圆圈（线宽3）

plt.imshow(test_img, cmap='gray')
plt.title("Matplotlib Test Image Output")
plt.axis("off")
plt.show()

### 2. 数据加载与初步探索（Notebook）
- **目的**：读入 CelebA 的 CSV 标注 + 部分图像，了解数据。
- **实现内容**：
  - 读取：
    - `list_attr_celeba.csv`（40 个属性）
    - `list_landmarks_celeba.csv`（5 个关键点）
    - `list_bbox_celeba.csv`（人脸框）
    - `list_eval_partition.csv`（train/val/test 划分）
  - 展示：
    - 数据量统计、属性分布条形图；
    - 随机几张脸 + 对应属性。
- **知识点**：图像数据集结构、标注文件组织方式。
- **主要库**：`pandas, cv2/PIL, matplotlib`。
---

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import numpy as np

# ================================
# 2.1 路径配置 —— 适配当前目录结构|
# 当前工作目录：dazuoye/notebooks
# 数据目录：    dazuoye/data/archive
# ================================
DATA_ROOT = os.path.abspath("../archive")  # 用相对路径定位数据目录，向上一级找到archive
IMG_DIR   = os.path.join(DATA_ROOT, "img_align_celeba")  # CelebA 对齐后的人脸图像目录

# CelebA 数据集的四个标注 CSV 文件路径
ATTR_PATH = os.path.join(DATA_ROOT, "list_attr_celeba.csv")  # 40个属性标注（Smiling, Male等）
BBOX_PATH = os.path.join(DATA_ROOT, "list_bbox_celeba.csv")  # 人脸框标注（x_1, y_1, width, height）
LAND_PATH = os.path.join(DATA_ROOT, "list_landmarks_align_celeba.csv")  # 5个关键点坐标（眼、鼻、嘴）
PART_PATH = os.path.join(DATA_ROOT, "list_eval_partition.csv")  # 训练/验证/测试集划分

print("=== 路径确认 ===")
for p in [DATA_ROOT, IMG_DIR, ATTR_PATH, BBOX_PATH, LAND_PATH, PART_PATH]:
    print(f"{p:60s} -> {'OK' if os.path.exists(p) else 'NOT FOUND'}")

In [ ]:
# CelebA 的这几个是逗号分隔的 CSV，不要再加 sep="\s+"

attr_df = pd.read_csv(ATTR_PATH)   # 40 个二值属性 + image_id，值为-1/1
bbox_df = pd.read_csv(BBOX_PATH)   # image_id, x_1, y_1, width, height
land_df = pd.read_csv(LAND_PATH)   # image_id + 10个关键点坐标（5个点 x 2坐标）
part_df = pd.read_csv(PART_PATH)   # image_id, partition（0=train, 1=val, 2=test）

print("属性表 shape:", attr_df.shape)
print("bbox   shape:", bbox_df.shape)
print("landm  shape:", land_df.shape)
print("划分表 shape:", part_df.shape)

display(attr_df.head())
display(bbox_df.head())
display(land_df.head())
display(part_df.head())

In [ ]:
# 看一眼列名确认一下
print(attr_df.columns.tolist())
print(bbox_df.columns.tolist())
print(land_df.columns.tolist())
print(part_df.columns.tolist())

# 为了统一命名，可以把 partition 列名改成 split（可选）
part_df = part_df.rename(columns={"partition": "split"})

# 合并四个表（以image_id为键），得到一个总表 full_df（56列）
full_df = (
    attr_df
    .merge(bbox_df, on="image_id")    # 合并bbox信息
    .merge(land_df, on="image_id")    # 合并关键点信息
    .merge(part_df, on="image_id")    # 合并train/val/test划分
)

print("合并后 shape:", full_df.shape)
display(full_df.head())

In [5]:
print("数据集划分统计（0=train, 1=val, 2=test）：")
split_counts = full_df["split"].value_counts().sort_index()
print(split_counts)


数据集划分统计（0=train, 1=val, 2=test）：
split
0    162770
1     19867
2     19962
Name: count, dtype: int64


In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

# ========= 路径配置 =========
# 原始人脸图像目录（与你项目一致）
IMG_DIR = IMG_DIR  # 若未定义，可改成 r"data/archive/img_align_celeba"

# 结果保存目录
RESULT_DIR = "results"
os.makedirs(RESULT_DIR, exist_ok=True)

# ========= 随机选一张图 =========
sample = full_df.sample(1, random_state=0).iloc[0]  # 用random_state保证可复现
img_path = os.path.join(IMG_DIR, sample["image_id"])
print("示例图片：", img_path)

# ========= 读取图像 =========
img = cv2.imread(img_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # BGR转RGB以便matplotlib正确显示

# ========= 可视化 =========
plt.figure(figsize=(5, 5))
plt.imshow(img)

# --- 画 bbox（人脸边界框） ---
x, y, w, h = sample[["x_1", "y_1", "width", "height"]]
plt.gca().add_patch(
    plt.Rectangle((x, y), w, h, fill=False, linewidth=2)  # 仅画边框，不填充
)

# --- 画五官关键点 ---
for lx, ly in [
    ("lefteye_x", "lefteye_y"),
    ("righteye_x", "righteye_y"),
    ("nose_x", "nose_y"),
    ("leftmouth_x", "leftmouth_y"),
    ("rightmouth_x", "rightmouth_y"),
]:
    plt.scatter(sample[lx], sample[ly], s=20)

plt.axis("off")
plt.title(f"Bounding box & landmarks\n{sample['image_id']}")

# ========= 保存 =========
save_path = os.path.join(
    RESULT_DIR,
    f"fig_bbox_landmarks_{sample['image_id'].replace('.jpg', '')}.png"
)

plt.savefig(save_path, dpi=200, bbox_inches="tight")
plt.show()

print("已保存到：", save_path)

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# ========= 读取图像 =========
img = cv2.imread(img_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
H, W = img.shape[:2]  # 获取图像高宽

# ========= 取 5 个关键点（从full_df的一行中提取） =========
pts = np.array([
    [sample["lefteye_x"],   sample["lefteye_y"]],    # 左眼
    [sample["righteye_x"],  sample["righteye_y"]],   # 右眼
    [sample["nose_x"],      sample["nose_y"]],       # 鼻子
    [sample["leftmouth_x"], sample["leftmouth_y"]],  # 左嘴角
    [sample["rightmouth_x"],sample["rightmouth_y"]], # 右嘴角
], dtype=float)

# ========= 用关键点计算 bbox（并扩展 30%）=========
x_min, y_min = pts.min(axis=0)  # 所有关键点的最小x,y
x_max, y_max = pts.max(axis=0)  # 所有关键点的最大x,y

w = x_max - x_min
h = y_max - y_min
p = 0.30  # 扩展比例（30%）

x0 = x_min - p * w  # 左边界向外扩展
y0 = y_min - p * h  # 上边界向外扩展
x1 = x_max + p * w  # 右边界向外扩展
y1 = y_max + p * h  # 下边界向外扩展

# clamp 到图像边界，防止越界
x0 = int(max(0, np.floor(x0)))
y0 = int(max(0, np.floor(y0)))
x1 = int(min(W - 1, np.ceil(x1)))
y1 = int(min(H - 1, np.ceil(y1)))

bbox_w = x1 - x0
bbox_h = y1 - y0

# ========= 论文级绘图（居中显示 bbox 区域）=========
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(img)

# 画红色bbox
ax.add_patch(
    plt.Rectangle((x0, y0), bbox_w, bbox_h,
                  fill=False, edgecolor="red", linewidth=2.5)
)

# landmarks语义分色：cyan=眼睛, yellow=鼻子, lime=嘴巴
ax.scatter(pts[0:2,0], pts[0:2,1], s=55, c="cyan",  edgecolors="black", linewidths=0.5, zorder=3)  # eyes
ax.scatter(pts[2,0],   pts[2,1],   s=55, c="yellow",edgecolors="black", linewidths=0.5, zorder=3)  # nose
ax.scatter(pts[3:5,0], pts[3:5,1], s=55, c="lime",  edgecolors="black", linewidths=0.5, zorder=3)  # mouth

# 关键：把视野缩放到 bbox 周围 → 人脸自动居中
m = int(0.15 * max(bbox_w, bbox_h))  # 额外留白（15%）
xl0 = max(0, x0 - m); xl1 = min(W, x1 + m)
yl0 = max(0, y0 - m); yl1 = min(H, y1 + m)

ax.set_xlim(xl0, xl1)
ax.set_ylim(yl1, yl0)   # 注意：imshow 的 y 轴向下，所以要反过来

ax.set_title("Face bounding box and landmarks", fontsize=11)
ax.axis("off")

save_path = os.path.join("results", "fig_bbox_landmarks.png")
plt.savefig(save_path, dpi=400, bbox_inches="tight")
plt.show()

print("论文图已保存：", save_path)

In [ ]:
import os
import matplotlib.pyplot as plt

# ========= 保存目录 =========
RESULT_DIR = "results"
os.makedirs(RESULT_DIR, exist_ok=True)

# ========= 选择要分析的属性 =========
attr_cols = ["Male", "Smiling", "Young", "Wearing_Hat", "Eyeglasses"]

# CelebA 属性是 -1 / 1 → 转为 0 / 1（replace映射）
attr_bin = full_df[attr_cols].replace({-1: 0, 1: 1})

# 计算每个属性的正例比例（均值即为正例占比）
pos_rates = attr_bin.mean().sort_values(ascending=False)
print("各属性正例比例：")
print(pos_rates)

# ========= 画图 =========
plt.figure(figsize=(6, 4))  # SCI 论文常用比例
pos_rates.plot(kind="bar", color="#4C72B0")

plt.ylabel("Positive Ratio", fontsize=11)
plt.xlabel("Attribute", fontsize=11)
plt.title("Distribution of Selected CelebA Attributes", fontsize=12)

plt.ylim(0, 1)
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.xticks(rotation=30)

plt.tight_layout()

# ========= 高清保存 =========
save_path = os.path.join(
    RESULT_DIR,
    "fig_attribute_distribution.png"
)

plt.savefig(
    save_path,
    dpi=400,              # SCI级清晰度：高于310即可
    bbox_inches="tight"
)

plt.show()

print("已保存高清图像：", save_path)

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

# ========= 路径配置 =========
IMG_DIR = Path("../archive/img_align_celeba")   # 按你当前 Notebook 所在路径
RESULT_DIR = Path("results")
RESULT_DIR.mkdir(exist_ok=True)

# ========= 随机抽样8张图 =========
samples = full_df.sample(8, random_state=42)
show_attrs = ["Smiling", "Male", "Young"]  # 标题中展示的属性

# ========= 可视化：2x4网格 =========
plt.figure(figsize=(10, 8))

for i, (_, row) in enumerate(samples.iterrows()):
    img_path = IMG_DIR / row["image_id"]
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.subplot(2, 4, i + 1)
    plt.imshow(img)
    plt.axis("off")

    # 属性值显示（将属性名和值拼成字符串）
    attr_text = ", ".join(
        [f"{col}={int(row[col])}" for col in show_attrs]
    )

    plt.title(
        f"{row['image_id']}\n{attr_text}",
        fontsize=8
    )

plt.tight_layout()

# ========= 高清保存 =========
save_path = RESULT_DIR / "fig_sample_faces_with_attributes.png"
plt.savefig(
    save_path,
    dpi=400,                 # SCI 级清晰度
    bbox_inches="tight"
)

plt.show()

print("已保存高清图像：", save_path)


### 3. 图像几何变换与对齐（Notebook）

- **目的**：做人脸对齐/裁剪/缩放，兼顾数据增强。
- **实现内容**：
  - 根据 bbox 裁剪人脸区域。
  - 根据两眼关键点估算旋转，做仿射对齐（旋正）。
  - 统一缩放到固定尺寸（如 128×128 或 224×224）。
  - 做若干几何增强示例：旋转、水平翻转等。
- **知识点**：仿射变换、插值、数据增强对模型的作用。
- **主要库**：`cv2.getRotationMatrix2D, cv2.warpAffine, cv2.resize, cv2.flip`。
### 4. 图像滤波与去噪（Notebook）

In [ ]:

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


# 如果上面已经定义过 IMG_DIR，这里可以注释掉
# IMG_DIR = Path("../data/archive/img_align_celeba")

def load_image(row):
    """根据 full_df 的一行记录读取原始图像(BGR)"""
    img_path = IMG_DIR / row["image_id"]
    img = cv2.imread(str(img_path))
    if img is None:
        raise ValueError(f"图像读取失败: {img_path}")
    return img


def show_rgb(img, title=None, ax=None):
    """安全显示 BGR 图像：自动处理空图像和坐标轴"""
    if ax is None:
        plt.figure(figsize=(4, 4))
        ax = plt.gca()
    if img is None or img.size == 0:  # 处理空图像的边界情况
        ax.text(0.5, 0.5, "空图像", ha="center", va="center")
        ax.axis("off")
        if title:
            ax.set_title(title)
        return ax
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))  # BGR转RGB
    ax.axis("off")
    if title:
        ax.set_title(title)
    return ax


# 让中文标题不报错（可选）
import matplotlib

matplotlib.rcParams['font.sans-serif'] = ['SimHei']
matplotlib.rcParams['axes.unicode_minus'] = False

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# ========= 结果保存目录 =========
RESULT_DIR = "results"
os.makedirs(RESULT_DIR, exist_ok=True)


def compute_bbox_from_landmarks(row, img_shape, expand_ratio=0.3):
    """
    利用 5 个关键点自动生成人脸 bbox，并适当扩边
    expand_ratio: 在关键点包围盒基础上进行扩展的比例
    """
    h, w = img_shape[:2]

    # 提取5个关键点的x坐标
    xs = np.array([
        row["lefteye_x"],
        row["righteye_x"],
        row["nose_x"],
        row["leftmouth_x"],
        row["rightmouth_x"],
    ], dtype=np.float32)

    # 提取5个关键点的y坐标
    ys = np.array([
        row["lefteye_y"],
        row["righteye_y"],
        row["nose_y"],
        row["leftmouth_y"],
        row["rightmouth_y"],
    ], dtype=np.float32)

    x_min, x_max = xs.min(), xs.max()
    y_min, y_max = ys.min(), ys.max()

    cx = (x_min + x_max) / 2  # 包围盒中心x
    cy = (y_min + y_max) / 2  # 包围盒中心y
    bw = (x_max - x_min) * (1 + expand_ratio)  # 扩展后宽度
    bh = (y_max - y_min) * (1 + expand_ratio * 1.2)  # 竖直方向略微多放大

    # np.clip限制在图像范围内
    x1 = int(np.clip(cx - bw / 2, 0, w - 1))
    y1 = int(np.clip(cy - bh / 2, 0, h - 1))
    x2 = int(np.clip(cx + bw / 2, x1 + 1, w))
    y2 = int(np.clip(cy + bh / 2, y1 + 1, h))

    return x1, y1, x2, y2


def crop_face_auto(img, row, expand_ratio=0.3):
    """根据关键点自动估计 bbox 并裁剪人脸区域"""
    x1, y1, x2, y2 = compute_bbox_from_landmarks(row, img.shape, expand_ratio)
    face = img[y1:y2, x1:x2].copy()  # 数组切片裁剪
    return face, (x1, y1, x2, y2)


# ========= 测试示例：原图 + 自动 bbox =========
sample = full_df.sample(1, random_state=0).iloc[0]
img_raw = load_image(sample)  # BGR格式
face_crop, (x1, y1, x2, y2) = crop_face_auto(img_raw, sample, expand_ratio=0.3)

# ========= 可视化 =========
fig, axes = plt.subplots(1, 2, figsize=(6, 3))

show_rgb(img_raw, "Original Image", ax=axes[0])

# 在原图上画自动 bbox（红色矩形框）
img_box = img_raw.copy()
cv2.rectangle(img_box, (x1, y1), (x2, y2), (0, 0, 255), 2)  # BGR红色，线宽2
show_rgb(img_box, "Auto-generated Bounding Box", ax=axes[1])

plt.tight_layout()

# ========= 高清保存 =========
save_path = os.path.join(
    RESULT_DIR,
    f"fig_auto_bbox_from_landmarks_{sample['image_id'].replace('.jpg', '')}.png"
)

plt.savefig(
    save_path,
    dpi=400,                 # SCI 级清晰度
    bbox_inches="tight"
)

plt.show()
print("已保存高清图像：", save_path)

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# ========= 结果保存目录 =========
RESULT_DIR = "results"
os.makedirs(RESULT_DIR, exist_ok=True)


def align_face_by_eyes(img, row, output_size=(128, 128), expand_ratio=0.3):
    """
    1. 基于五官关键点自动生成人脸 bbox 并裁剪
    2. 根据双眼连线估计旋转角度并进行仿射对齐
    3. resize 到统一尺寸
    """
    face, (x1, y1, x2, y2) = crop_face_auto(img, row, expand_ratio)
    fh, fw = face.shape[:2]

    # 裁剪后关键点坐标（相对于 face 左上角）
    lx = row["lefteye_x"] - x1
    ly = row["lefteye_y"] - y1
    rx = row["righteye_x"] - x1
    ry = row["righteye_y"] - y1

    # 计算双眼连线角度（arctan2返回弧度，转为角度）
    dy = ry - ly
    dx = rx - lx
    angle = np.degrees(np.arctan2(dy, dx))  # 正值=逆时针
    center = ((lx + rx) / 2.0, (ly + ry) / 2.0)  # 旋转中心=双眼中点

    # 仿射旋转（取负角使脸部旋正）
    M = cv2.getRotationMatrix2D(center, -angle, 1.0)  # 缩放因子1.0
    face_aligned = cv2.warpAffine(
        face, M, (fw, fh), flags=cv2.INTER_LINEAR  # INTER_LINEAR双线性插值
    )

    # resize 到统一尺寸
    face_resized = cv2.resize(
        face_aligned, output_size, interpolation=cv2.INTER_LINEAR
    )

    return face_resized


# ========= 效果演示 =========
sample = full_df.sample(1, random_state=1).iloc[0]
img_raw = load_image(sample)  # BGR

face_crop, _ = crop_face_auto(img_raw, sample, expand_ratio=0.3)
face_aligned = align_face_by_eyes(
    img_raw, sample, output_size=(128, 128), expand_ratio=0.3
)

# ========= 可视化：三步流水线 =========
fig, axes = plt.subplots(1, 3, figsize=(9, 3))

show_rgb(img_raw, "Original Image", ax=axes[0])
show_rgb(face_crop, "Auto BBox Crop", ax=axes[1])
show_rgb(face_aligned, "Aligned & Resized (128×128)", ax=axes[2])

plt.tight_layout()

# ========= 高清保存 =========
save_path = os.path.join(
    RESULT_DIR,
    f"fig_face_alignment_pipeline_{sample['image_id'].replace('.jpg', '')}.png"
)

plt.savefig(
    save_path,
    dpi=400,                 # SCI 级清晰度
    bbox_inches="tight"
)

plt.show()
print("已保存高清图像：", save_path)

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

# ========= 结果保存目录 =========
RESULT_DIR = "results"
os.makedirs(RESULT_DIR, exist_ok=True)


def augment_geometric(img, angle=15):
    """
    对齐后人脸进行几何增强：
    - 小角度旋转
    - 水平翻转
    """
    h, w = img.shape[:2]
    center = (w / 2, h / 2)  # 旋转中心=图像中心

    # 旋转：getRotationMatrix2D(center, 角度, 缩放)
    M_rot = cv2.getRotationMatrix2D(center, angle, 1.0)
    img_rot = cv2.warpAffine(
        img, M_rot, (w, h), flags=cv2.INTER_LINEAR
    )

    # 水平翻转：flipCode=1表示水平翻转（0=垂直, -1=both）
    img_flip = cv2.flip(img, 1)

    return img_rot, img_flip


# ========= 示例演示 =========
sample = full_df.sample(1, random_state=2).iloc[0]
img_raw = load_image(sample)

face_aligned = align_face_by_eyes(
    img_raw, sample, output_size=(128, 128)
)
rot_img, flip_img = augment_geometric(face_aligned, angle=15)

# ========= 可视化 =========
fig, axes = plt.subplots(1, 3, figsize=(9, 3))

show_rgb(face_aligned, "Aligned Face", ax=axes[0])
show_rgb(rot_img, "Rotation (+15°)", ax=axes[1])
show_rgb(flip_img, "Horizontal Flip", ax=axes[2])

plt.tight_layout()

# ========= 高清保存 =========
save_path = os.path.join(
    RESULT_DIR,
    f"fig_geometric_augmentation_{sample['image_id'].replace('.jpg', '')}.png"
)

plt.savefig(
    save_path,
    dpi=400,                 # SCI 级清晰度
    bbox_inches="tight"
)

plt.show()
print("已保存高清图像：", save_path)

In [ ]:
import os
import matplotlib.pyplot as plt

# ========= 结果保存目录 =========
RESULT_DIR = "results"
os.makedirs(RESULT_DIR, exist_ok=True)

# ========= 示例样本：完整4步处理流水线 =========
sample = full_df.sample(1, random_state=42).iloc[0]
img_raw = load_image(sample)  # BGR

face_crop, _ = crop_face_auto(img_raw, sample, expand_ratio=0.3)
face_aligned = align_face_by_eyes(img_raw, sample, output_size=(128, 128))
rot_img, flip_img = augment_geometric(face_aligned, angle=20)

# ========= 可视化：4步流水线 =========
fig, axes = plt.subplots(1, 4, figsize=(12, 3))

show_rgb(img_raw, "Original Image", ax=axes[0])
show_rgb(face_crop, "Auto BBox Crop", ax=axes[1])
show_rgb(face_aligned, "Aligned & Resized (128×128)", ax=axes[2])
show_rgb(rot_img, "Rotation (+20°)", ax=axes[3])

plt.tight_layout()

# ========= 高清保存 =========
save_path = os.path.join(
    RESULT_DIR,
    f"fig_geometric_processing_pipeline_{sample['image_id'].replace('.jpg', '')}.png"
)

plt.savefig(
    save_path,
    dpi=400,                 # SCI 级分辨率
    bbox_inches="tight"
)

plt.show()
print("已保存高清图像：", save_path)

- **目的**：演示空域滤波：降噪 + 锐化。
- **实现内容**：
  - 高斯滤波 / 均值滤波 / 中值滤波对比。
  - 拉普拉斯或自定义卷积核做锐化。
  - 展示“原图 vs 模糊 vs 锐化”对比。
- **知识点**：卷积、低通/高通滤波，噪声-细节的权衡。
- **主要库**：`cv2.GaussianBlur, cv2.medianBlur, cv2.filter2D`。

---

In [ ]:
import os
import matplotlib.pyplot as plt

# ========= 结果保存目录 =========
RESULT_DIR = "results"
os.makedirs(RESULT_DIR, exist_ok=True)

# ========= 随机挑一张样本 =========
sample = full_df.sample(1, random_state=123).iloc[0]

# 原图 + 对齐后人脸
img_raw = load_image(sample)  # BGR
face_aligned = align_face_by_eyes(
    img_raw, sample, output_size=(128, 128)
)

# ========= 可视化：原图 vs 对齐图 =========
fig, axes = plt.subplots(1, 2, figsize=(6, 3))

show_rgb(img_raw, "Original Image", ax=axes[0])
show_rgb(face_aligned, "Aligned Face (128×128)", ax=axes[1])

plt.tight_layout()

# ========= 高清保存 =========
save_path = os.path.join(
    RESULT_DIR,
    f"fig_original_vs_aligned_{sample['image_id'].replace('.jpg', '')}.png"
)

plt.savefig(
    save_path,
    dpi=400,                 # SCI 级分辨率
    bbox_inches="tight"
)

plt.show()
print("已保存高清图像：", save_path)

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

# ========= 结果保存目录 =========
RESULT_DIR = "results"
os.makedirs(RESULT_DIR, exist_ok=True)

# ========= 随机挑一张样本 =========
sample = full_df.sample(1, random_state=123).iloc[0]

# 原图 + 对齐人脸
img_raw = load_image(sample)   # BGR
face_aligned = align_face_by_eyes(
    img_raw, sample, output_size=(128, 128)
)

# ========= 空域滤波：三种常用滤波器对比 =========
# 高斯滤波：权重按高斯分布，平滑同时保留一定结构
gauss = cv2.GaussianBlur(face_aligned, ksize=(5, 5), sigmaX=1.0)

# 均值滤波（box filter）：邻域像素简单取平均，模糊效果最强
mean = cv2.blur(face_aligned, ksize=(5, 5))

# 中值滤波：取邻域中位数，对椒盐噪声特别有效
median = cv2.medianBlur(face_aligned, ksize=5)

# ========= 可视化：4列对比 =========
fig, axes = plt.subplots(1, 4, figsize=(10, 3))

show_rgb(face_aligned, "Aligned Face", ax=axes[0])
show_rgb(gauss, "Gaussian Blur", ax=axes[1])
show_rgb(mean, "Mean Filter", ax=axes[2])
show_rgb(median, "Median Filter", ax=axes[3])

plt.tight_layout()

# ========= 高清保存 =========
save_path = os.path.join(
    RESULT_DIR,
    f"fig_spatial_filtering_comparison_{sample['image_id'].replace('.jpg', '')}.png"
)

plt.savefig(
    save_path,
    dpi=400,                 # SCI 级清晰度
    bbox_inches="tight"
)

plt.show()
print("已保存高清图像：", save_path)

In [ ]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt

# ========= 结果保存目录 =========
RESULT_DIR = "results"
os.makedirs(RESULT_DIR, exist_ok=True)

# ========= 随机挑一张样本 =========
sample = full_df.sample(1, random_state=123).iloc[0]

# 原图 + 对齐后人脸
img_raw = load_image(sample)   # BGR
face_aligned = align_face_by_eyes(
    img_raw, sample, output_size=(128, 128)
)

# ========= 锐化：拉普拉斯风格卷积核 =========
# 中心为5、周围为-1的核：增强边缘对比度（原图 + 拉普拉斯边缘 = 锐化）
sharp_kernel = np.array([
    [0, -1,  0],
    [-1,  5, -1],
    [0, -1,  0]
], dtype=np.float32)

# filter2D(ddepth=-1)：输出深度与输入相同
sharp = cv2.filter2D(
    face_aligned, ddepth=-1, kernel=sharp_kernel
)

# ========= 可视化 =========
fig, axes = plt.subplots(1, 2, figsize=(6, 3))

show_rgb(face_aligned, "Aligned Face", ax=axes[0])
show_rgb(sharp, "Sharpened Image", ax=axes[1])

plt.tight_layout()

# ========= 高清保存 =========
save_path = os.path.join(
    RESULT_DIR,
    f"fig_image_sharpening_{sample['image_id'].replace('.jpg', '')}.png"
)

plt.savefig(
    save_path,
    dpi=400,                 # SCI 级清晰度
    bbox_inches="tight"
)

plt.show()
print("已保存高清图像：", save_path)

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

# ========= 结果保存目录 =========
RESULT_DIR = "results"
os.makedirs(RESULT_DIR, exist_ok=True)

# ========= 随机挑一张样本 =========
sample = full_df.sample(1, random_state=123).iloc[0]

# 原图 + 对齐后人脸
img_raw = load_image(sample)   # BGR
face_aligned = align_face_by_eyes(
    img_raw, sample, output_size=(128, 128)
)

# ========= Unsharp Mask（反锐化掩膜） =========
# 先做较强的高斯模糊
blur_strong = cv2.GaussianBlur(
    face_aligned, (0, 0), sigmaX=2.0  # sigmaX=2.0，较强模糊
)

# Unsharp Mask公式：1.5 * 原图 - 0.5 * 模糊图
# addWeighted(alpha=1.5, beta=-0.5, gamma=0) 等价于 1.5*src1 + (-0.5)*src2
unsharp = cv2.addWeighted(
    face_aligned, 1.5, blur_strong, -0.5, 0
)

# ========= 可视化 =========
fig, axes = plt.subplots(1, 3, figsize=(9, 3))

show_rgb(face_aligned, "Aligned Face", ax=axes[0])
show_rgb(blur_strong, "Strong Gaussian Blur", ax=axes[1])
show_rgb(unsharp, "Unsharp Mask Sharpening", ax=axes[2])

plt.tight_layout()

# ========= 高清保存 =========
save_path = os.path.join(
    RESULT_DIR,
    f"fig_unsharp_mask_{sample['image_id'].replace('.jpg', '')}.png"
)

plt.savefig(
    save_path,
    dpi=400,                 # SCI 级分辨率
    bbox_inches="tight"
)

plt.show()
print("已保存高清图像：", save_path)

In [ ]:
# ===== Figure 6: 空域滤波与增强效果对比（2x2网格论文图） =====
import matplotlib.pyplot as plt

# 假设你前面已经算好了这些变量：
# face_aligned, gauss, mean, median, sharp, unsharp

fig, axes = plt.subplots(2, 2, figsize=(7.5, 6))  # 控制尺寸，防止过宽

# (a) 原始对齐人脸
axes[0, 0].imshow(cv2.cvtColor(face_aligned, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title("(a) Aligned face", fontsize=10)
axes[0, 0].axis("off")

# (b) 高斯滤波（空域滤波代表）
axes[0, 1].imshow(cv2.cvtColor(gauss, cv2.COLOR_BGR2RGB))
axes[0, 1].set_title("(b) Gaussian filtering", fontsize=10)
axes[0, 1].axis("off")

# (c) 拉普拉斯锐化
axes[1, 0].imshow(cv2.cvtColor(sharp, cv2.COLOR_BGR2RGB))
axes[1, 0].set_title("(c) Laplacian-based sharpening", fontsize=10)
axes[1, 0].axis("off")

# (d) Unsharp Mask增强
axes[1, 1].imshow(cv2.cvtColor(unsharp, cv2.COLOR_BGR2RGB))
axes[1, 1].set_title("(d) Unsharp mask enhancement", fontsize=10)
axes[1, 1].axis("off")

plt.tight_layout()

# ===== SCI 级保存 =====
plt.savefig(
    "./results/Figure_6_spatial_filtering_and_enhancement.png",
    dpi=400,
    bbox_inches="tight",
    pad_inches=0.02  # 论文友好微边距
)

plt.show()

- **目的**：演示空域滤波：降噪 + 锐化。
- **实现内容**：
  - 高斯滤波 / 均值滤波 / 中值滤波对比。
  - 拉普拉斯或自定义卷积核做锐化。
  - 展示“原图 vs 模糊 vs 锐化”对比。
- **知识点**：卷积、低通/高通滤波，噪声-细节的权衡。
- **主要库**：`cv2.GaussianBlur, cv2.medianBlur, cv2.filter2D`。

---
### 5. 直方图处理与图像增强（Notebook）
- **目的**：用直方图均衡化改善对比度，减轻光照差异。
- **实现内容**：
  - 灰度图直方图均衡化。
  - 彩色图：只对亮度通道做 HE 或 CLAHE。
  - 绘制均衡前后直方图和图像效果。
- **知识点**：灰度变换、CDF、全局 vs 自适应均衡。
- **主要库**：`cv2.equalizeHist, cv2.createCLAHE, skimage.exposure`。
---

In [ ]:

from skimage import exposure  # 后面用于自适应直方图均衡化CLAHE

sample = full_df.sample(1, random_state=2025).iloc[0]
img_raw = load_image(sample)
face = align_face_by_eyes(img_raw, sample, output_size=(128, 128))

fig, ax = plt.subplots(1, 2, figsize=(6, 3))
show_rgb(img_raw, "原图", ax=ax[0])
show_rgb(face, "对齐人脸(128×128)", ax=ax[1])
plt.tight_layout()
plt.show()

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

# ========= 结果保存目录 =========
RESULT_DIR = "results"
os.makedirs(RESULT_DIR, exist_ok=True)

# ========= 随机样本（与上一节保持一致） =========
sample = full_df.sample(1, random_state=2025).iloc[0]

# 原图 + 对齐后人脸
img_raw = load_image(sample)   # BGR
face = align_face_by_eyes(
    img_raw, sample, output_size=(128, 128)
)

# ========= 灰度直方图均衡化 =========
face_gray = cv2.cvtColor(face, cv2.COLOR_BGR2GRAY)  # BGR转灰度

# equalizeHist：全局直方图均衡化，将像素分布拉伸到均匀分布
face_gray_he = cv2.equalizeHist(face_gray)

# ========= 可视化：2x2网格 =========
fig, axes = plt.subplots(2, 2, figsize=(8, 5))

# 原灰度图
axes[0, 0].imshow(face_gray, cmap="gray")
axes[0, 0].set_title("Grayscale Image")
axes[0, 0].axis("off")

# 均衡化后灰度图
axes[0, 1].imshow(face_gray_he, cmap="gray")
axes[0, 1].set_title("Histogram Equalization")
axes[0, 1].axis("off")

# 原图灰度直方图（ravel展平为1D，64个bin，log对数尺度便于观察）
axes[1, 0].hist(
    face_gray.ravel(),
    bins=64,
    range=(0, 256),
    color="black",
    alpha=0.9
)
axes[1, 0].set_title("Histogram (Original)")
axes[1, 0].set_xlim(0, 255)
axes[1, 0].set_yscale("log")  # 对数尺度让高频和低频都可见
axes[1, 0].set_ylabel("Pixel count")
axes[1, 0].grid(alpha=0.3)

# 均衡化后直方图
axes[1, 1].hist(
    face_gray_he.ravel(),
    bins=64,
    range=(0, 256),
    color="black",
    alpha=0.9
)
axes[1, 1].set_title("Histogram (Equalized)")
axes[1, 1].set_xlim(0, 255)
axes[1, 1].set_yscale("log")
axes[1, 1].grid(alpha=0.3)

# ========= 高清保存 =========
save_path = os.path.join(
    RESULT_DIR,
    f"fig_histogram_equalization_gray_{sample['image_id'].replace('.jpg', '')}.png"
)

plt.savefig(
    save_path,
    dpi=400,                 # SCI 级清晰度
    bbox_inches="tight"
)

plt.show()
print("已保存高清图像：", save_path)

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

# ========= 结果保存目录 =========
RESULT_DIR = "results"
os.makedirs(RESULT_DIR, exist_ok=True)

# ========= 随机样本 =========
sample = full_df.sample(1, random_state=2025).iloc[0]

# 原图 + 对齐后人脸
img_raw = load_image(sample)   # BGR
face = align_face_by_eyes(
    img_raw, sample, output_size=(128, 128)
)

# ========= 彩色图像亮度直方图均衡 =========
# BGR -> YCrCb颜色空间：Y=亮度, Cr/Cb=色度
ycrcb = cv2.cvtColor(face, cv2.COLOR_BGR2YCrCb)
Y, Cr, Cb = cv2.split(ycrcb)  # 分离三个通道

# 仅对亮度通道Y做HE，色度通道保持不变（避免色彩失真）
Y_he = cv2.equalizeHist(Y)

# 合并并转回 BGR
ycrcb_he = cv2.merge([Y_he, Cr, Cb])
face_color_he = cv2.cvtColor(ycrcb_he, cv2.COLOR_YCrCb2BGR)

# ========= 彩色图像对比显示 =========
fig1, axes1 = plt.subplots(1, 2, figsize=(6, 3))

show_rgb(face, "Original Color Image", ax=axes1[0])
show_rgb(face_color_he, "Luminance Equalized (Y Channel)", ax=axes1[1])

plt.tight_layout()

save_path_img = os.path.join(
    RESULT_DIR,
    f"fig_color_luminance_equalization_{sample['image_id'].replace('.jpg', '')}.png"
)

plt.savefig(
    save_path_img,
    dpi=400,                 # SCI 级清晰度
    bbox_inches="tight"
)

plt.show()
print("已保存高清图像：", save_path_img)

# ========= 亮度直方图对比（改进版） =========
fig2, axes2 = plt.subplots(1, 2, figsize=(8, 3))

# 原始 Y 通道直方图
axes2[0].hist(
    Y.ravel(),
    bins=64,                 # 减少bin让分布更清晰
    range=(0, 256),
    color="black",
    alpha=0.9
)
axes2[0].set_title("Histogram of Y Channel (Original)")
axes2[0].set_xlim(0, 255)
axes2[0].set_yscale("log")   # 对数尺度
axes2[0].set_ylabel("Pixel count")
axes2[0].grid(alpha=0.3)

# 均衡化后 Y 通道直方图
axes2[1].hist(
    Y_he.ravel(),
    bins=64,
    range=(0, 256),
    color="black",
    alpha=0.9
)
axes2[1].set_title("Histogram of Y Channel (Equalized)")
axes2[1].set_xlim(0, 255)
axes2[1].set_yscale("log")
axes2[1].grid(alpha=0.3)

plt.tight_layout()

save_path_hist = os.path.join(
    RESULT_DIR,
    f"fig_color_luminance_histogram_{sample['image_id'].replace('.jpg', '')}.png"
)

plt.savefig(
    save_path_hist,
    dpi=400,
    bbox_inches="tight"
)

plt.show()
print("已保存高清直方图：", save_path_hist)

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

# ========= 结果保存目录 =========
RESULT_DIR = "results"
os.makedirs(RESULT_DIR, exist_ok=True)

# ========= 随机样本 =========
sample = full_df.sample(1, random_state=2025).iloc[0]

# 原图 + 对齐后人脸
img_raw = load_image(sample)   # BGR
face = align_face_by_eyes(
    img_raw, sample, output_size=(128, 128)
)

# ========= YCrCb 颜色空间转换 =========
ycrcb = cv2.cvtColor(face, cv2.COLOR_BGR2YCrCb)
Y, Cr, Cb = cv2.split(ycrcb)

# ========= 全局 HE =========
Y_he = cv2.equalizeHist(Y)
ycrcb_he = cv2.merge([Y_he, Cr, Cb])
face_color_he = cv2.cvtColor(ycrcb_he, cv2.COLOR_YCrCb2BGR)

# ========= CLAHE（限制对比度自适应直方图均衡化） =========
# clipLimit=2.0：限制对比度放大倍数，防止过增强
# tileGridSize=(8,8)：将图像分成8x8个块，每块独立均衡化
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
Y_clahe = clahe.apply(Y)  # 对Y通道应用CLAHE
ycrcb_clahe = cv2.merge([Y_clahe, Cr, Cb])
face_color_clahe = cv2.cvtColor(ycrcb_clahe, cv2.COLOR_YCrCb2BGR)

# ========= 彩色图像对比：原图 vs 全局HE vs CLAHE =========
fig1, axes1 = plt.subplots(1, 3, figsize=(9, 3))

show_rgb(face, "Original Color Image", ax=axes1[0])
show_rgb(face_color_he, "Global HE (Y Channel)", ax=axes1[1])
show_rgb(face_color_clahe, "CLAHE (Y Channel)", ax=axes1[2])

plt.tight_layout()

save_path_img = os.path.join(
    RESULT_DIR,
    f"fig_he_vs_clahe_color_{sample['image_id'].replace('.jpg', '')}.png"
)

plt.savefig(
    save_path_img,
    dpi=400,                 # SCI 级分辨率
    bbox_inches="tight"
)

plt.show()
print("已保存高清图像：", save_path_img)

# ========= 亮度直方图对比 =========
fig2, axes2 = plt.subplots(1, 3, figsize=(9, 3))

axes2[0].hist(Y.ravel(), bins=256, range=(0, 256))
axes2[0].set_title("Histogram of Y (Original)")

axes2[1].hist(Y_he.ravel(), bins=256, range=(0, 256))
axes2[1].set_title("Histogram of Y (Global HE)")

axes2[2].hist(Y_clahe.ravel(), bins=256, range=(0, 256))
axes2[2].set_title("Histogram of Y (CLAHE)")

plt.tight_layout()

save_path_hist = os.path.join(
    RESULT_DIR,
    f"fig_he_vs_clahe_histogram_{sample['image_id'].replace('.jpg', '')}.png"
)

plt.savefig(
    save_path_hist,
    dpi=400,                 # SCI 级分辨率
    bbox_inches="tight"
)

plt.show()
print("已保存高清直方图：", save_path_hist)

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

# ========= 结果保存目录 =========
RESULT_DIR = "results"
os.makedirs(RESULT_DIR, exist_ok=True)

# ========= 随机样本 =========
sample = full_df.sample(1, random_state=2025).iloc[0]

# 原图 + 对齐后人脸
img_raw = load_image(sample)   # BGR
face = align_face_by_eyes(
    img_raw, sample, output_size=(128, 128)
)

# ========= YCrCb 颜色空间 =========
ycrcb = cv2.cvtColor(face, cv2.COLOR_BGR2YCrCb)
Y, Cr, Cb = cv2.split(ycrcb)

# ========= 全局 HE =========
Y_he = cv2.equalizeHist(Y)
ycrcb_he = cv2.merge([Y_he, Cr, Cb])
face_color_he = cv2.cvtColor(ycrcb_he, cv2.COLOR_YCrCb2BGR)

# ========= CLAHE =========
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
Y_clahe = clahe.apply(Y)
ycrcb_clahe = cv2.merge([Y_clahe, Cr, Cb])
face_color_clahe = cv2.cvtColor(ycrcb_clahe, cv2.COLOR_YCrCb2BGR)

# ========= 彩色图像对比 =========
fig1, axes1 = plt.subplots(1, 3, figsize=(9, 3))

show_rgb(face, "Original Color Image", ax=axes1[0])
show_rgb(face_color_he, "Global HE (Y Channel)", ax=axes1[1])
show_rgb(face_color_clahe, "CLAHE (Y Channel)", ax=axes1[2])

plt.tight_layout()

save_path_img = os.path.join(
    RESULT_DIR,
    f"fig_he_vs_clahe_color_{sample['image_id'].replace('.jpg', '')}.png"
)

plt.savefig(
    save_path_img,
    dpi=400,
    bbox_inches="tight"
)

plt.show()
print("已保存高清图像：", save_path_img)

# ========= 亮度直方图对比（改进版：统一风格参数） =========
fig2, axes2 = plt.subplots(1, 3, figsize=(9, 3))

# 统一hist_kwargs保证三张直方图可比
hist_kwargs = dict(
    bins=64,                 # 256减少到64个bin，分布更清晰
    range=(0, 256),
    color="black",
    alpha=0.9
)

# 原始
axes2[0].hist(Y.ravel(), **hist_kwargs)
axes2[0].set_title("Histogram of Y (Original)")
axes2[0].set_xlim(0, 255)
axes2[0].set_yscale("log")    # 对数尺度便于观察分布差异
axes2[0].set_ylabel("Pixel count")
axes2[0].grid(alpha=0.3)

# Global HE
axes2[1].hist(Y_he.ravel(), **hist_kwargs)
axes2[1].set_title("Histogram of Y (Global HE)")
axes2[1].set_xlim(0, 255)
axes2[1].set_yscale("log")
axes2[1].grid(alpha=0.3)

# CLAHE
axes2[2].hist(Y_clahe.ravel(), **hist_kwargs)
axes2[2].set_title("Histogram of Y (CLAHE)")
axes2[2].set_xlim(0, 255)
axes2[2].set_yscale("log")
axes2[2].grid(alpha=0.3)

plt.tight_layout()

save_path_hist = os.path.join(
    RESULT_DIR,
    f"fig_he_vs_clahe_histogram_{sample['image_id'].replace('.jpg', '')}.png"
)

plt.savefig(
    save_path_hist,
    dpi=400,
    bbox_inches="tight"
)

plt.show()
print("已保存高清直方图：", save_path_hist)

In [ ]:

# skimage自适应直方图均衡化：输入要求[0,1]范围的浮点图像
face_gray_float = face_gray / 255.0  # uint8[0,255]转float[0,1]
face_gray_adapt = exposure.equalize_adapthist(face_gray_float, clip_limit=0.03)  # clip_limit控制对比度

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(face_gray, cmap="gray")
axes[0].set_title("原灰度")
axes[0].axis("off")

axes[1].imshow(face_gray_he, cmap="gray")
axes[1].set_title("OpenCV HE")
axes[1].axis("off")

axes[2].imshow(face_gray_adapt, cmap="gray")
axes[2].set_title("skimage 自适应 HE")
axes[2].axis("off")

plt.tight_layout()
plt.show()

### 6. 图像分割与形态学处理（Notebook）
- **目的**：尝试把人脸前景从背景里分出来。
- **实现内容**：
  - Otsu 阈值法生成粗糙的人脸掩膜。
  - 形态学开/闭运算去噪和填洞。
  - 展示：“原图、初始掩膜、形态学后掩膜”。
  - 可选：对某几张图用 K-means 做简易颜色分割。
- **知识点**：阈值分割、腐蚀/膨胀、开闭运算。
- **主要库**：`cv2.threshold, cv2.morphologyEx, skimage.morphology`。
---

In [ ]:
sample = full_df.sample(1, random_state=666).iloc[0]
img_raw = load_image(sample)
face = align_face_by_eyes(img_raw, sample, output_size=(128, 128))

face_gray = cv2.cvtColor(face, cv2.COLOR_BGR2GRAY)

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
show_rgb(face, "对齐人脸", ax=axes[0])
axes[1].imshow(face_gray, cmap="gray")
axes[1].set_title("灰度图")
axes[1].axis("off")

plt.tight_layout()

# ================= SCI 级别保存 =================
plt.savefig(
    "./results/face_alignment_gray.png",
    dpi=400,                 # 400 PPI
    bbox_inches="tight",     # 去掉多余白边
    pad_inches=0.02          # 论文友好微边距
)

plt.show()

In [ ]:
# Otsu 自动阈值：自动计算最佳阈值进行二值化
# ret=最佳阈值, mask_raw=0/255二值图
ret, mask_raw = cv2.threshold(
    face_gray, 0, 255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU  # Otsu算法自动选择阈值
)

print("Otsu 自动阈值 =", ret)

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(face_gray, cmap="gray")
axes[0].set_title("灰度图")
axes[0].axis("off")

axes[1].imshow(mask_raw, cmap="gray")
axes[1].set_title("初始掩膜 (Otsu)")
axes[1].axis("off")

plt.tight_layout()

# ================= SCI 级别保存 =================
plt.savefig(
    "./results/otsu_mask.png",
    dpi=400,                 # 400 PPI / DPI
    bbox_inches="tight",     # 去白边
    pad_inches=0.02          # 论文安全边距
)

plt.show()

In [ ]:
# getStructuringElement：创建形态学结构元素（核），MORPH_ELLIPSE为椭圆形
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))

# 开运算（MORPH_OPEN）：先腐蚀再膨胀，去掉小白点噪声
mask_open = cv2.morphologyEx(mask_raw, cv2.MORPH_OPEN, kernel, iterations=1)

# 闭运算（MORPH_CLOSE）：先膨胀再腐蚀，填补小黑洞
mask_clean = cv2.morphologyEx(mask_open, cv2.MORPH_CLOSE, kernel, iterations=2)

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(mask_raw, cmap="gray")
axes[0].set_title("初始掩膜")
axes[0].axis("off")

axes[1].imshow(mask_open, cmap="gray")
axes[1].set_title("开运算后")
axes[1].axis("off")

axes[2].imshow(mask_clean, cmap="gray")
axes[2].set_title("开+闭 后掩膜")
axes[2].axis("off")

plt.tight_layout()

# ================= SCI 论文级保存 =================
plt.savefig(
    "./results/morphology_cleaning.png",
    dpi=400,                 # 400 PPI / DPI
    bbox_inches="tight",     # 去除多余白边
    pad_inches=0.02          # 期刊安全边距
)

plt.show()

In [ ]:
# 把 0/255 掩膜变成 0/1 的三通道（用于逐像素乘法）
mask_3c = cv2.cvtColor(mask_clean, cv2.COLOR_GRAY2BGR) // 255

# 前景 = face * mask（只保留掩膜内的像素）
foreground = face * mask_3c
# 背景用灰色填充，便于对比显示分割效果
background_gray = cv2.cvtColor(face, cv2.COLOR_BGR2GRAY)
background_gray = cv2.cvtColor(background_gray, cv2.COLOR_GRAY2BGR)
bg_with_hole = background_gray * (1 - mask_3c)  # 反向掩膜：背景区域
segmented = foreground + bg_with_hole  # 前景 + 背景 = 完整分割结果

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
show_rgb(face, "原图", ax=axes[0])
axes[1].imshow(mask_clean, cmap="gray")
axes[1].set_title("最终掩膜")
axes[1].axis("off")
show_rgb(segmented, "掩膜分割效果", ax=axes[2])

plt.tight_layout()

# ================= SCI 论文级保存 =================
plt.savefig(
    "./results/face_segmentation_result.png",
    dpi=400,                 # 400 PPI / DPI
    bbox_inches="tight",     # 去除白边
    pad_inches=0.02          # 期刊安全边距
)

plt.show()

In [ ]:
# K-means颜色分割：将图像像素聚成K类颜色
Z = face.reshape((-1, 3)).astype(np.float32)  # reshape为(N,3)的像素列表

K = 3  # 聚类数：分成3种颜色
# 终止条件：epsilon=1.0或最大迭代20次
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
# attempts=5：运行5次取最优结果；KMEANS_RANDOM_CENTERS：随机初始化中心
ret, labels, centers = cv2.kmeans(
    Z, K, None, criteria, 5, cv2.KMEANS_RANDOM_CENTERS
)

centers = np.uint8(centers)  # 将聚类中心转为uint8颜色值
seg_kmeans = centers[labels.flatten()].reshape(face.shape)  # 用聚类结果替换原图像素

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
show_rgb(face, "原图", ax=axes[0])
show_rgb(seg_kmeans, "K-means 颜色分割 (K=3)", ax=axes[1])

plt.tight_layout()

# ================= SCI 论文级保存 =================
plt.savefig(
    "./results/kmeans_color_segmentation.png",
    dpi=400,                 # 400 PPI / DPI
    bbox_inches="tight",     # 去除白边
    pad_inches=0.02          # 期刊安全边距
)

plt.show()

- **目的**：提取轮廓结构（下巴、眼睛、眼镜框等）。
- **实现内容**：
  - 计算 Sobel X/Y、梯度幅值图。
  - Canny 边缘检测。
  - 将边缘叠加到原图上展示。
- **知识点**：一阶/二阶导、Sobel、Canny 各步骤。
- **主要库**：`cv2.Sobel, cv2.Canny, skimage.filters, skimage.feature.canny`。

---

In [ ]:
sample = full_df.sample(1, random_state=777).iloc[0]
img_raw = load_image(sample)
face = align_face_by_eyes(img_raw, sample, output_size=(128, 128))
face_gray = cv2.cvtColor(face, cv2.COLOR_BGR2GRAY)

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
show_rgb(face, "对齐人脸", ax=axes[0])
axes[1].imshow(face_gray, cmap="gray")
axes[1].set_title("灰度图")
axes[1].axis("off")

plt.tight_layout()

# ================= SCI 论文级保存 =================
plt.savefig(
    "./results/aligned_face_gray_sample777.png",
    dpi=400,                 # 400 PPI / DPI
    bbox_inches="tight",     # 去白边
    pad_inches=0.02          # 期刊安全边距
)

plt.show()

In [ ]:
# Sobel X / Y（浮点结果，CV_64F防止负值溢出）
sobel_x = cv2.Sobel(face_gray, cv2.CV_64F, 1, 0, ksize=3)  # dx=1:计算x方向梯度
sobel_y = cv2.Sobel(face_gray, cv2.CV_64F, 0, 1, ksize=3)  # dy=1:计算y方向梯度

# 梯度幅值 = sqrt(sobel_x^2 + sobel_y^2)
grad_mag = np.sqrt(sobel_x ** 2 + sobel_y ** 2)
grad_mag = np.uint8(255 * grad_mag / grad_mag.max())  # 归一化到 0-255

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(np.abs(sobel_x), cmap="gray")  # abs取绝对值显示
axes[0].set_title("Sobel X")
axes[0].axis("off")

axes[1].imshow(np.abs(sobel_y), cmap="gray")
axes[1].set_title("Sobel Y")
axes[1].axis("off")

axes[2].imshow(grad_mag, cmap="gray")
axes[2].set_title("梯度幅值")
axes[2].axis("off")

plt.tight_layout()

# ================= SCI 论文级保存 =================
plt.savefig(
    "./results/sobel_gradient_visualization.png",
    dpi=400,                 # 400 PPI / DPI
    bbox_inches="tight",     # 去除多余白边
    pad_inches=0.02          # 期刊安全边距
)

plt.show()

In [ ]:
# Canny边缘检测：双阈值+NMS（非极大值抑制）
# threshold1=50(低阈值), threshold2=150(高阈值)
edges = cv2.Canny(face_gray, threshold1=50, threshold2=150)

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(face_gray, cmap="gray")
axes[0].set_title("灰度图")
axes[0].axis("off")

axes[1].imshow(edges, cmap="gray")
axes[1].set_title("Canny 边缘")
axes[1].axis("off")

plt.tight_layout()

# ================= SCI 论文级保存 =================
plt.savefig(
    "./results/canny_edge_detection.png",
    dpi=400,                 # 400 PPI / DPI
    bbox_inches="tight",     # 去除多余白边
    pad_inches=0.02          # 期刊安全边距
)

plt.show()

In [ ]:
from skimage.feature import canny

edges_ski = canny(face_gray, sigma=1.5)  # skimage返回bool数组（True/False），sigma控制高斯平滑程度

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(edges_ski, cmap="gray")
axes[0].set_title("skimage Canny")
axes[0].axis("off")

axes[1].imshow(edges, cmap="gray")
axes[1].set_title("OpenCV Canny")
axes[1].axis("off")

plt.tight_layout()

# ================= SCI 论文级保存 =================
plt.savefig(
    "./results/canny_skimage_vs_opencv.png",
    dpi=400,                 # 400 PPI / DPI
    bbox_inches="tight",     # 去白边
    pad_inches=0.02          # 期刊安全边距
)

In [ ]:
edges_color = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)

overlay = face.copy()
overlay[edges > 0] = (0, 255, 0)  # BGR格式：(0,255,0)为绿色，将边缘像素设为绿色

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
show_rgb(face, "原图", ax=axes[0])
show_rgb(overlay, "边缘叠加 (绿色)", ax=axes[1])

plt.tight_layout()

# ================= SCI 论文级保存 =================
plt.savefig(
    "./results/canny_edge_overlay.png",
    dpi=400,                 # 400 PPI / DPI
    bbox_inches="tight",     # 去除多余白边
    pad_inches=0.02          # 期刊安全边距
)

plt.show()


### 8. 特征提取（Notebook）
- **目的**：把图像变成特征向量，连接到“机器学习世界”。
- **实现内容**：
  - 基于分割掩膜的形状特征：面积、周长等。
  - 纹理特征：LBP / HOG。
  - 角点特征：Harris 角点或 ORB 关键点。
  - 可选：抽几张图用预训练 CNN 提取特征向量（当作对比）。
- **知识点**：形状特征、纹理特征、局部特征点的概念；图像→向量。
- **主要库**：
  - `skimage.feature.local_binary_pattern, skimage.feature.hog`
  - `- `cv2.cornerHarris, cv2.ORB_create` 等。

In [ ]:
import cv2
import numpy as np
from skimage.feature import local_binary_pattern, hog


def get_face_and_mask(row, size=(128, 128)):
    """对齐人脸 + Otsu分割 + 开闭运算，返回 face, mask_clean"""
    img_raw = load_image(row)
    face = align_face_by_eyes(img_raw, row, output_size=size)
    gray = cv2.cvtColor(face, cv2.COLOR_BGR2GRAY)

    # Otsu 阈值自动二值化
    _, mask = cv2.threshold(
        gray, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )
    # 形态学开运算去噪 + 闭运算填洞
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    mask_open = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask_clean = cv2.morphologyEx(mask_open, cv2.MORPH_CLOSE, kernel, iterations=2)
    return face, gray, mask_clean


sample = full_df.sample(1, random_state=8).iloc[0]
face, face_gray, mask_clean = get_face_and_mask(sample)

# findContours：提取轮廓
# RETR_EXTERNAL：只找最外层轮廓；CHAIN_APPROX_SIMPLE：压缩水平/垂直/对角线段
contours, _ = cv2.findContours(mask_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# 计算形状特征：面积和周长
area = 0.0
perimeter = 0.0
for cnt in contours:
    area += cv2.contourArea(cnt)        # 轮廓面积
    perimeter += cv2.arcLength(cnt, True)  # 轮廓周长（True=闭合）

print("形状特征：")
print("  面积 (pixels):", area)
print("  周长 (pixels):", perimeter)
print("  面积占比:", area / (mask_clean.shape[0] * mask_clean.shape[1]))

# 可视化：原图 + 掩膜 + 轮廓叠加
contour_vis = face.copy()
cv2.drawContours(contour_vis, contours, -1, (0, 255, 0), 1)  # 绿色轮廓，线宽1

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
show_rgb(face, "对齐人脸", ax=axes[0])
axes[1].imshow(mask_clean, cmap="gray")
axes[1].set_title("掩膜")
axes[1].axis("off")
show_rgb(contour_vis, "轮廓叠加", ax=axes[2])

plt.tight_layout()

# ================= SCI 论文级保存 =================
plt.savefig(
    "./results/shape_features_contour.png",
    dpi=400,                 # 400 PPI / DPI
    bbox_inches="tight",     # 去白边
    pad_inches=0.02          # 期刊安全边距
)

plt.show()

In [ ]:
# LBP（局部二值模式）纹理特征：P=8个采样点, R=1像素半径
P = 8
R = 1
# method="uniform"：统一模式，将所有非统一模式合并为1个bin，减少特征维度
lbp = local_binary_pattern(face_gray, P, R, method="uniform")

# LBP直方图作为特征向量
n_bins = P + 2  # uniform模式下理论bins数 = P+2（P个统一模式 + 1个非统一 + 1个overflow）
hist_lbp, _ = np.histogram(
    lbp.ravel(),        # 展平为1D
    bins=n_bins,
    range=(0, n_bins),
    density=True        # 归一化为概率分布
)

print("LBP 特征维度:", hist_lbp.shape)
print("LBP 特征前 10 维:", hist_lbp[:10])

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].imshow(lbp, cmap="gray")
axes[0].set_title("LBP 图")
axes[0].axis("off")

axes[1].bar(np.arange(n_bins), hist_lbp)
axes[1].set_title("LBP 直方图特征")

plt.tight_layout()

# ================= SCI 论文级保存 =================
plt.savefig(
    "./results/lbp_texture_features.png",
    dpi=400,                 # 400 PPI / DPI
    bbox_inches="tight",     # 去除多余白边
    pad_inches=0.02          # 期刊安全边距
)

plt.show()

In [ ]:
# HOG（方向梯度直方图）特征提取
# orientations=9：每个cell的梯度方向分成9个bin
# pixels_per_cell=(8,8)：每个cell大小为8x8像素
# cells_per_block=(2,2)：每个block包含2x2个cell
# block_norm="L2-Hys"：L2范数归一化+截断
hog_vec, hog_vis = hog(
    face_gray,
    orientations=9,
    pixels_per_cell=(8, 8),
    cells_per_block=(2, 2),
    visualize=True,          # 同时返回可视化图
    block_norm="L2-Hys"
)

print("HOG 特征维度:", hog_vec.shape)

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].imshow(face_gray, cmap="gray")
axes[0].set_title("灰度图")
axes[0].axis("off")

axes[1].imshow(hog_vis, cmap="gray")
axes[1].set_title("HOG 可视化")
axes[1].axis("off")

plt.tight_layout()

# ================= SCI 论文级保存 =================
plt.savefig(
    "./results/hog_feature_visualization.png",
    dpi=400,                 # 400 PPI / DPI
    bbox_inches="tight",     # 去除多余白边
    pad_inches=0.02          # 期刊安全边距
)

plt.show()

In [ ]:
# Harris 角点检测
gray_float = np.float32(face_gray)  # Harris需要float32输入
# cornerHarris(blockSize=2, ksize=3, k=0.04)
# blockSize=2：自相关矩阵的邻域大小；ksize=Sobel核大小；k=Harris方程的自由参数
harris = cv2.cornerHarris(gray_float, 2, 3, 0.04)
harris = cv2.dilate(harris, None)  # 膨胀使角点更明显

# 阈值筛选：harris响应值 > th * 最大响应值 的位置为角点
th = 0.05  # 阈值比例，可尝试0.05~0.1
corner_vis = face.copy()
corner_vis[harris > th * harris.max()] = [255, 0, 0]  # BGR红色标记角点

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
show_rgb(face, "原图", ax=axes[0])
show_rgb(corner_vis, "Harris 角点 (红色)", ax=axes[1])

plt.tight_layout()

# ================= SCI 论文级保存 =================
plt.savefig(
    "./results/harris_corner_detection.png",
    dpi=400,                 # 400 PPI / DPI
    bbox_inches="tight",     # 去除多余白边
    pad_inches=0.02          # 期刊安全边距
)

plt.show()

In [ ]:
# Harris 角点检测（v2，降低阈值以检测更多角点）
gray_float = np.float32(face_gray)
harris = cv2.cornerHarris(gray_float, blockSize=2, ksize=3, k=0.04)
harris = cv2.dilate(harris, None)

# 阈值降低到0.01，检测更多显著角点
corner_vis = face.copy()
corner_vis[harris > 0.01 * harris.max()] = [0, 0, 255]  # BGR红色标记

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
show_rgb(face, "原图", ax=axes[0])
show_rgb(corner_vis, "Harris 角点 (红色)", ax=axes[1])

plt.tight_layout()

# ================= SCI 论文级保存 =================
plt.savefig(
    "./results/harris_corner_detection_v2.png",
    dpi=400,                 # 400 PPI / DPI
    bbox_inches="tight",     # 去除多余白边
    pad_inches=0.02          # 期刊安全边距
)

plt.show()

In [ ]:

import torch
import torchvision.models as models
import torchvision.transforms as T

# 加载预训练ResNet18（ImageNet权重）
resnet18 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
resnet18.eval()  # 设为评估模式（关闭Dropout和BatchNorm的训练行为）

# 去掉最后的FC分类层，只保留卷积特征提取部分
# ResNet18最后是Linear(512, 1000)，去掉后输出512维特征
feature_extractor = torch.nn.Sequential(*list(resnet18.children())[:-1])

# ImageNet标准预处理流水线
transform_cnn = T.Compose([
    T.ToPILImage(),
    T.Resize((224, 224)),               # ResNet输入要求224x224
    T.ToTensor(),                        # 转为Tensor，归一化到[0,1]
    T.Normalize(mean=[0.485, 0.456, 0.406],   # ImageNet均值
                std=[0.229, 0.224, 0.225]),     # ImageNet标准差
])

# face是BGR格式，需要转成RGB再输入CNN
face_rgb = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
inp = transform_cnn(face_rgb).unsqueeze(0)  # 增加batch维度：[1,3,224,224]

with torch.no_grad():  # 不计算梯度，节省内存和时间
    feat = feature_extractor(inp)  # 输出shape: [1, 512, 1, 1]
feat_vec = feat.view(-1).cpu().numpy()  # 展平为512维向量

print("ResNet18 特征维度:", feat_vec.shape)
print("前 10 维:", feat_vec[:10])

### 保存为txt格式

In [ ]:
import json
from pathlib import Path

# ---------- 1) 自动定位 all.ipynb ----------
def find_all_ipynb():
    """从当前目录向上搜索all.ipynb文件"""
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        cand1 = p / "all.ipynb"
        cand2 = p / "notebooks" / "all.ipynb"
        if cand1.exists():
            return cand1
        if cand2.exists():
            return cand2
    return None

all_nb = find_all_ipynb()
assert all_nb is not None, "未找到 all.ipynb（请确认项目目录结构里有 notebooks/all.ipynb）"

# ---------- 2) 读取 ipynb JSON ----------
with open(all_nb, "r", encoding="utf-8") as f:
    nb = json.load(f)

# ---------- 3) 收集所有 code cell ----------
code_blocks = []
for i, cell in enumerate(nb.get("cells", [])):
    if cell.get("cell_type") == "code":
        src = "".join(cell.get("source", [])).strip()
        if src:
            code_blocks.append(f"# ===== Cell {i} =====\n{src}\n")

# ---------- 4) 输出到 notebooks/results ----------
out_dir = all_nb.parent / "results"
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / "all_code.txt"
with open(out_path, "w", encoding="utf-8") as f:
    f.write("\n\n".join(code_blocks))

print(f"读取：{all_nb}")
print(f"导出 code cells：{len(code_blocks)} 个")
print(f"保存到：{out_path}")

In [ ]:
import json
from pathlib import Path

# =====================================================
# 1. 定位项目根目录 & all.ipynb
# =====================================================
def find_project_root_and_all_ipynb():
    """向上搜索同时包含 notebooks/all.ipynb 和 src/ 目录的项目根目录"""
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        nb = p / "notebooks" / "all.ipynb"
        src = p / "src"
        if nb.exists() and src.exists():
            return p, nb, src
    return None, None, None

project_root, all_nb, src_dir = find_project_root_and_all_ipynb()
assert all_nb is not None, "未找到 notebooks/all.ipynb"
assert src_dir is not None, "未找到 src 目录"

# =====================================================
# 2. 读取 all.ipynb 的 code cells
# =====================================================
with open(all_nb, "r", encoding="utf-8") as f:
    nb = json.load(f)

sections = []

sections.append("##############################\n"
                "# NOTEBOOK: all.ipynb\n"
                "##############################\n")

for i, cell in enumerate(nb.get("cells", [])):
    if cell.get("cell_type") == "code":
        src = "".join(cell.get("source", [])).strip()
        if src:
            sections.append(
                f"\n# ===== Notebook Cell {i} =====\n{src}\n"
            )

# =====================================================
# 3. 读取 src 下所有 .py 文件
# =====================================================
sections.append("\n\n##############################\n"
                "# SOURCE FILES: src/*.py\n"
                "##############################\n")

py_files = sorted(src_dir.glob("*.py"))

for py in py_files:
    with open(py, "r", encoding="utf-8") as f:
        code = f.read().strip()
    sections.append(
        f"\n# ===== File: src/{py.name} =====\n{code}\n"
    )

# =====================================================
# 4. 输出到 notebooks/results
# =====================================================
out_dir = all_nb.parent / "results"
out_dir.mkdir(exist_ok=True)

out_path = out_dir / "all_bundle.txt"
with open(out_path, "w", encoding="utf-8") as f:
    f.write("\n".join(sections))

print("导出完成")
print(f"Notebook: {all_nb}")
print(f"Python files: {[p.name for p in py_files]}")
print(f"输出文件: {out_path}")